### **Loading CIFAR-10**

How does the Latent Space Dimension (10 vs. 100) affect image fidelity and diversity?

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Image transformation
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

100%|██████████| 170M/170M [00:03<00:00, 48.6MB/s]


### **The Generator**

In [3]:
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input is latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # State size: 256 x 4 x 4
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # State size: 128 x 8 x 8
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # State size: 64 x 16 x 16
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
            # Final size: 3 x 32 x 32
        )

    def forward(self, x):
        return self.main(x)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1)

### **The Discriminator**

In [4]:
LATENT_DIM = 100
netG = Generator(LATENT_DIM).to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Training Loop
for epoch in range(25): # 25-50 epochs recommended
    for i, (data, _) in enumerate(dataloader):
        netD.zero_grad()
        real = data.to(device)
        batch_size = real.size(0)
        label = torch.full((batch_size,), 1.0, device=device)
        output = netD(real)
        errD_real = criterion(output, label)
        errD_real.backward()

        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(0.0)
        output = netD(fake.detach())
        errD_fake = criterion(output, label)
        errD_fake.backward()
        optimizerD.step()

        # 2. Update Generator: maximize log(D(G(z)))
        netG.zero_grad()
        label.fill_(1.0)
        output = netD(fake)
        errG = criterion(output, label)
        errG.backward()
        optimizerG.step()

    print(f'Epoch [{epoch}/25] Loss_D: {errD_real+errD_fake:.4f} Loss_G: {errG:.4f}')

Epoch [0/25] Loss_D: 0.3127 Loss_G: 4.2698
Epoch [1/25] Loss_D: 0.3129 Loss_G: 3.0475
Epoch [2/25] Loss_D: 0.4782 Loss_G: 3.3417
Epoch [3/25] Loss_D: 0.5782 Loss_G: 2.1540
Epoch [4/25] Loss_D: 0.4229 Loss_G: 2.6498
Epoch [5/25] Loss_D: 0.6284 Loss_G: 2.5313
Epoch [6/25] Loss_D: 0.6125 Loss_G: 1.6962
Epoch [7/25] Loss_D: 0.3935 Loss_G: 2.4197
Epoch [8/25] Loss_D: 0.5591 Loss_G: 2.8777
Epoch [9/25] Loss_D: 0.3699 Loss_G: 1.7698
Epoch [10/25] Loss_D: 0.4214 Loss_G: 2.8958
Epoch [11/25] Loss_D: 0.3522 Loss_G: 3.1848
Epoch [12/25] Loss_D: 0.3347 Loss_G: 2.9565
Epoch [13/25] Loss_D: 0.5765 Loss_G: 1.6030
Epoch [14/25] Loss_D: 0.3660 Loss_G: 2.3709
Epoch [15/25] Loss_D: 0.3256 Loss_G: 3.4209
Epoch [16/25] Loss_D: 0.2211 Loss_G: 3.3811
Epoch [17/25] Loss_D: 0.2985 Loss_G: 2.5731
Epoch [18/25] Loss_D: 0.3302 Loss_G: 2.6589
Epoch [19/25] Loss_D: 0.6317 Loss_G: 1.7666
Epoch [20/25] Loss_D: 0.2808 Loss_G: 2.5448
Epoch [21/25] Loss_D: 0.3090 Loss_G: 2.6803
Epoch [22/25] Loss_D: 0.4026 Loss_G: 2.059

In [5]:
LATENT_DIM = 10
netG = Generator(LATENT_DIM).to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Training Loop
for epoch in range(25):
    for i, (data, _) in enumerate(dataloader):
        netD.zero_grad()
        real = data.to(device)
        batch_size = real.size(0)
        label = torch.full((batch_size,), 1.0, device=device)
        output = netD(real)
        errD_real = criterion(output, label)
        errD_real.backward()

        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(0.0)
        output = netD(fake.detach())
        errD_fake = criterion(output, label)
        errD_fake.backward()
        optimizerD.step()

        # 2. Update Generator: maximize log(D(G(z)))
        netG.zero_grad()
        label.fill_(1.0)
        output = netD(fake)
        errG = criterion(output, label)
        errG.backward()
        optimizerG.step()

    print(f'Epoch [{epoch}/25] Loss_D: {errD_real+errD_fake:.4f} Loss_G: {errG:.4f}')

Epoch [0/25] Loss_D: 0.6716 Loss_G: 2.0554
Epoch [1/25] Loss_D: 0.6980 Loss_G: 3.0229
Epoch [2/25] Loss_D: 0.8005 Loss_G: 4.7887
Epoch [3/25] Loss_D: 0.6977 Loss_G: 2.9074
Epoch [4/25] Loss_D: 0.6182 Loss_G: 3.1696
Epoch [5/25] Loss_D: 0.7246 Loss_G: 1.9845
Epoch [6/25] Loss_D: 0.7347 Loss_G: 2.1764
Epoch [7/25] Loss_D: 0.8413 Loss_G: 4.5099
Epoch [8/25] Loss_D: 0.4888 Loss_G: 3.5032
Epoch [9/25] Loss_D: 0.8013 Loss_G: 1.6879
Epoch [10/25] Loss_D: 0.4231 Loss_G: 2.7045
Epoch [11/25] Loss_D: 0.6182 Loss_G: 1.8629
Epoch [12/25] Loss_D: 0.5515 Loss_G: 3.0469
Epoch [13/25] Loss_D: 0.5282 Loss_G: 3.2182
Epoch [14/25] Loss_D: 0.4032 Loss_G: 2.6781
Epoch [15/25] Loss_D: 0.4591 Loss_G: 2.6306
Epoch [16/25] Loss_D: 0.3091 Loss_G: 2.5146
Epoch [17/25] Loss_D: 0.6705 Loss_G: 3.5962
Epoch [18/25] Loss_D: 0.4423 Loss_G: 2.7531
Epoch [19/25] Loss_D: 0.4227 Loss_G: 2.4980
Epoch [20/25] Loss_D: 0.4619 Loss_G: 2.9025
Epoch [21/25] Loss_D: 0.2882 Loss_G: 3.6442
Epoch [22/25] Loss_D: 0.5172 Loss_G: 3.956

### **Training Loop**

In [6]:
import torchvision.utils as vutils
if epoch % 5 == 0:
    with torch.no_grad():
        fake_display = netG(torch.randn(64, LATENT_DIM, 1, 1, device=device))
        vutils.save_image(fake_display.detach(),
                          f'fake_samples_epoch_{epoch}_dim_{LATENT_DIM}.png',
                          normalize=True)

**Visual Results and Model Saving**

In [7]:
torch.save(netG.state_dict(), 'gan_generator.pth')